<a href="https://colab.research.google.com/github/Estevamfla/b3-financial-intelligence/blob/main/01_analise_financeira_petrobras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import requests
import zipfile
from io import BytesIO

ano = 2025

url = (
    "https://dados.cvm.gov.br/dados/"
    f"CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
)

print("Baixando dados da CVM...")

resposta = requests.get(url, timeout=120)
resposta.raise_for_status()

with zipfile.ZipFile(BytesIO(resposta.content)) as arquivo:
    nomes = arquivo.namelist()

print("Download concluído!")
print(f"Quantidade de arquivos: {len(nomes)}")

for nome in nomes[:10]:
    print(nome)


Baixando dados da CVM...
Download concluído!
Quantidade de arquivos: 19
dfp_cia_aberta_2025.csv
dfp_cia_aberta_BPA_con_2025.csv
dfp_cia_aberta_BPA_ind_2025.csv
dfp_cia_aberta_BPP_con_2025.csv
dfp_cia_aberta_BPP_ind_2025.csv
dfp_cia_aberta_composicao_capital_2025.csv
dfp_cia_aberta_DFC_MD_con_2025.csv
dfp_cia_aberta_DFC_MD_ind_2025.csv
dfp_cia_aberta_DFC_MI_con_2025.csv
dfp_cia_aberta_DFC_MI_ind_2025.csv


In [2]:

import pandas as pd
import zipfile
from io import BytesIO

# Abrir o ZIP baixado na etapa anterior
with zipfile.ZipFile(BytesIO(resposta.content)) as zip_cvm:

    # Localizar a DRE consolidada de 2025
    arquivo_dre = next(
        nome for nome in zip_cvm.namelist()
        if nome.endswith("DRE_con_2025.csv")
    )

    # Ler o CSV
    with zip_cvm.open(arquivo_dre) as arquivo:
        dre = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            dtype={"CNPJ_CIA": str, "CD_CONTA": str},
            low_memory=False
        )

# Filtrar o exercício atual e a versão mais recente
dre = dre[dre["ORDEM_EXERC"] == "ÚLTIMO"].copy()

ultima_versao = dre.groupby(
    ["CNPJ_CIA", "DT_REFER"]
)["VERSAO"].transform("max")

dre = dre[dre["VERSAO"] == ultima_versao]

# Selecionar informações principais
colunas = [
    "DENOM_CIA",
    "DT_FIM_EXERC",
    "CD_CONTA",
    "DS_CONTA",
    "VL_CONTA",
    "ESCALA_MOEDA"
]

dre = dre[colunas]

print("Total de registros:", len(dre))
print("Total de empresas:", dre["DENOM_CIA"].nunique())

display(dre.head(10))


Total de registros: 15490
Total de empresas: 440


,DENOM_CIA,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ESCALA_MOEDA
1,BCO BRASIL S.A.,2025-12-31,3.01,Receitas de Intermediação Financeira,319462104.0,MIL
3,BCO BRASIL S.A.,2025-12-31,3.01.01,Receita de Juros,319462104.0,MIL
5,BCO BRASIL S.A.,2025-12-31,3.02,Despesas de Intermediação Financeira,-218451386.0,MIL
7,BCO BRASIL S.A.,2025-12-31,3.02.01,Despesa de Juros,-218451386.0,MIL
9,BCO BRASIL S.A.,2025-12-31,3.03,Resultado Bruto de Intermediação Financeira,101010718.0,MIL
11,BCO BRASIL S.A.,2025-12-31,3.04,Outras Despesas e Receitas Operacionais,-95397082.0,MIL
13,BCO BRASIL S.A.,2025-12-31,3.04.01,Despesa de Provisão para Perda Esperada para R...,-66633285.0,MIL
15,BCO BRASIL S.A.,2025-12-31,3.04.02,Receitas de Prestação de Serviços,34714379.0,MIL
17,BCO BRASIL S.A.,2025-12-31,3.04.03,Despesas com Pessoal,-29178666.0,MIL
19,BCO BRASIL S.A.,2025-12-31,3.04.04,Outras Despesas de Administrativas,-9455529.0,MIL


In [4]:

# Criar uma cópia dos dados originais
dre_tratada = dre.copy()

# Converter valores para números
dre_tratada["VL_CONTA"] = pd.to_numeric(
    dre_tratada["VL_CONTA"],
    errors="coerce"
)

# Normalizar a escala monetária
fatores = {
    "MIL": 1000,
    "REAL": 1,
    "UNIDADE": 1
}

escala = (
    dre_tratada["ESCALA_MOEDA"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Verificar escalas desconhecidas
desconhecidas = set(escala.dropna()) - set(fatores)

if desconhecidas:
    raise ValueError(
        f"Escalas não reconhecidas: {desconhecidas}"
    )

# Converter todos os valores para reais
dre_tratada["VALOR_REAIS"] = (
    dre_tratada["VL_CONTA"] * escala.map(fatores)
)

print("Conversão realizada com sucesso!")

display(
    dre_tratada[
        ["DENOM_CIA", "DS_CONTA", "VALOR_REAIS"]
    ].head(10)
)


Conversão realizada com sucesso!


,DENOM_CIA,DS_CONTA,VALOR_REAIS
1,BCO BRASIL S.A.,Receitas de Intermediação Financeira,3.194621e+11
3,BCO BRASIL S.A.,Receita de Juros,3.194621e+11
5,BCO BRASIL S.A.,Despesas de Intermediação Financeira,-2.184514e+11
7,BCO BRASIL S.A.,Despesa de Juros,-2.184514e+11
9,BCO BRASIL S.A.,Resultado Bruto de Intermediação Financeira,1.010107e+11
11,BCO BRASIL S.A.,Outras Despesas e Receitas Operacionais,-9.539708e+10
13,BCO BRASIL S.A.,Despesa de Provisão para Perda Esperada para R...,-6.663328e+10
15,BCO BRASIL S.A.,Receitas de Prestação de Serviços,3.471438e+10
17,BCO BRASIL S.A.,Despesas com Pessoal,-2.917867e+10
19,BCO BRASIL S.A.,Outras Despesas de Administrativas,-9.455529e+09


In [5]:

petrobras = dre_tratada[
    dre_tratada["DENOM_CIA"].str.contains(
        "PETROBRAS",
        case=False,
        na=False
    )
].copy()

print("Registros encontrados:", len(petrobras))

display(
    petrobras[
        ["DENOM_CIA", "CD_CONTA", "DS_CONTA", "VALOR_REAIS"]
    ].head(20)
)


Registros encontrados: 43


,DENOM_CIA,CD_CONTA,DS_CONTA,VALOR_REAIS
17559,PETROLEO BRASILEIRO S.A. PETROBRAS,3.01,Receita de Venda de Bens e/ou Serviços,4.975490e+11
17561,PETROLEO BRASILEIRO S.A. PETROBRAS,3.02,Custo dos Bens e/ou Serviços Vendidos,-2.605510e+11
17563,PETROLEO BRASILEIRO S.A. PETROBRAS,3.03,Resultado Bruto,2.369980e+11
17565,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04,Despesas/Receitas Operacionais,-9.137000e+10
17567,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04.01,Despesas com Vendas,-2.895400e+10
17569,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04.02,Despesas Gerais e Administrativas,-1.080200e+10
17571,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04.03,Perdas pela Não Recuperabilidade de Ativos,0.000000e+00
17573,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04.04,Outras Receitas Operacionais,0.000000e+00
17575,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04.05,Outras Despesas Operacionais,-5.137200e+10
17577,PETROLEO BRASILEIRO S.A. PETROBRAS,3.04.05.01,Tributárias,-4.241000e+09


In [6]:

# Selecionar as três contas principais da DRE
contas = ["3.01", "3.02", "3.03"]

indicadores = petrobras[
    petrobras["CD_CONTA"].isin(contas)
].copy()

# Verificar se cada conta aparece uma única vez
for conta in contas:
    quantidade = (indicadores["CD_CONTA"] == conta).sum()

    if quantidade != 1:
        raise ValueError(
            f"Conta {conta}: encontrados {quantidade} registros."
        )

# Extrair os valores
valores = indicadores.set_index("CD_CONTA")["VALOR_REAIS"]

receita = valores["3.01"]
custo = valores["3.02"]
lucro_bruto = valores["3.03"]

# Calcular margem bruta
margem_bruta = (lucro_bruto / receita) * 100

# Validar a consistência contábil
diferenca = receita + custo - lucro_bruto

print("ANÁLISE FINANCEIRA - PETROBRAS 2025")
print("-" * 45)

print(f"Receita líquida: R$ {receita:,.2f}")
print(f"Custos: R$ {custo:,.2f}")
print(f"Lucro bruto: R$ {lucro_bruto:,.2f}")

print("-" * 45)
print(f"MARGEM BRUTA: {margem_bruta:.2f}%")
print(f"Diferença de conciliação: R$ {diferenca:,.2f}")


ANÁLISE FINANCEIRA - PETROBRAS 2025
---------------------------------------------
Receita líquida: R$ 497,549,000,000.00
Custos: R$ -260,551,000,000.00
Lucro bruto: R$ 236,998,000,000.00
---------------------------------------------
MARGEM BRUTA: 47.63%
Diferença de conciliação: R$ 0.00


In [7]:

# Verificar as contas finais da DRE da Petrobras
contas_finais = petrobras[
    petrobras["CD_CONTA"].str.match(
        r"^3\.(09|10|11)(\.|$)",
        na=False
    )
]

display(
    contas_finais[
        ["CD_CONTA", "DS_CONTA", "VALOR_REAIS"]
    ]
)


,CD_CONTA,DS_CONTA,VALOR_REAIS
17617,3.09,Resultado Líquido das Operações Continuadas,1.106050e+11
17619,3.10,Resultado Líquido de Operações Descontinuadas,0.000000e+00
17621,3.10.01,Lucro/Prejuízo Líquido das Operações Descontin...,0.000000e+00
17623,3.10.02,Ganhos/Perdas Líquidas sobre Ativos de Operaçõ...,0.000000e+00
17625,3.11,Lucro/Prejuízo Consolidado do Período,1.106050e+11
17627,3.11.01,Atribuído a Sócios da Empresa Controladora,1.101290e+11
17629,3.11.02,Atribuído a Sócios Não Controladores,4.760000e+08


In [8]:

# Identificar o lucro líquido consolidado
conta_lucro = petrobras[
    petrobras["CD_CONTA"] == "3.11"
]

if len(conta_lucro) != 1:
    raise ValueError(
        "Esperado exatamente um registro da conta 3.11"
    )

lucro_liquido = conta_lucro["VALOR_REAIS"].iloc[0]

# Utilizar a receita obtida anteriormente
margem_liquida = (lucro_liquido / receita) * 100

# Conferir a distribuição do lucro consolidado
contas_atribuicao = petrobras[
    petrobras["CD_CONTA"].isin(["3.11.01", "3.11.02"])
]

if len(contas_atribuicao) == 2:
    diferenca = (
        contas_atribuicao["VALOR_REAIS"].sum()
        - lucro_liquido
    )
    print(f"Diferença da atribuição: R$ {diferenca:,.2f}")

print("\nPETROBRAS - RESULTADOS 2025")
print("-" * 45)
print(f"Receita líquida: R$ {receita:,.2f}")
print(f"Lucro bruto: R$ {lucro_bruto:,.2f}")
print(f"Lucro líquido: R$ {lucro_liquido:,.2f}")
print(f"Margem bruta: {margem_bruta:.2f}%")
print(f"Margem líquida: {margem_liquida:.2f}%")


Diferença da atribuição: R$ 0.00

PETROBRAS - RESULTADOS 2025
---------------------------------------------
Receita líquida: R$ 497,549,000,000.00
Lucro bruto: R$ 236,998,000,000.00
Lucro líquido: R$ 110,605,000,000.00
Margem bruta: 47.63%
Margem líquida: 22.23%


In [9]:

import pandas as pd

resumo = pd.DataFrame([{
    "Empresa": "Petrobras",
    "Ano": 2025,
    "Receita_Liquida": receita,
    "Lucro_Bruto": lucro_bruto,
    "Lucro_Liquido": lucro_liquido,
    "Margem_Bruta_%": round(margem_bruta, 2),
    "Margem_Liquida_%": round(margem_liquida, 2)
}])

display(resumo)

# Criar arquivo para utilização futura
resumo.to_csv(
    "indicadores_petrobras_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo CSV criado com sucesso!")


,Empresa,Ano,Receita_Liquida,Lucro_Bruto,Lucro_Liquido,Margem_Bruta_%,Margem_Liquida_%
0,Petrobras,2025,4.975490e+11,2.369980e+11,1.106050e+11,47.63,22.23


Arquivo CSV criado com sucesso!


In [11]:

import pandas as pd
import zipfile
from io import BytesIO

# Função para ler os balanços da CVM
def ler_balanco(tipo):

    nome = f"dfp_cia_aberta_{tipo}_con_2025.csv"

    # Abrir o ZIP dentro da função
    with zipfile.ZipFile(BytesIO(resposta.content)) as zip_cvm:

        with zip_cvm.open(nome) as arquivo:

            df = pd.read_csv(
                arquivo,
                sep=";",
                encoding="latin1",
                dtype={
                    "CNPJ_CIA": str,
                    "CD_CONTA": str
                },
                low_memory=False
            )

    # Filtrar exercício atual
    df = df[df["ORDEM_EXERC"] == "ÚLTIMO"].copy()

    # Selecionar versão mais recente
    versoes = df.groupby(
        ["CNPJ_CIA", "DT_REFER"]
    )["VERSAO"].transform("max")

    df = df[df["VERSAO"] == versoes].copy()

    # Converter valores para reais
    fatores = {
        "MIL": 1000,
        "UNIDADE": 1,
        "REAL": 1
    }

    escala = (
        df["ESCALA_MOEDA"]
        .str.strip()
        .str.upper()
    )

    if not escala.isin(fatores).all():
        raise ValueError("Escala monetária desconhecida")

    df["VALOR_REAIS"] = (
        pd.to_numeric(df["VL_CONTA"], errors="raise")
        * escala.map(fatores)
    )

    return df


# Carregar os dois demonstrativos
bpa = ler_balanco("BPA")
bpp = ler_balanco("BPP")

print("Balanços carregados com sucesso!")
print("Registros do ativo:", len(bpa))
print("Registros do passivo:", len(bpp))


Balanços carregados com sucesso!
Registros do ativo: 29828
Registros do passivo: 51040


In [12]:

# Identificar a Petrobras pelo CNPJ
cnpj_petrobras = "33.000.167/0001-01"

# Buscar o Ativo Circulante
ativo = bpa[
    (bpa["CNPJ_CIA"] == cnpj_petrobras) &
    (bpa["CD_CONTA"] == "1.01")
]

# Buscar o Passivo Circulante
passivo = bpp[
    (bpp["CNPJ_CIA"] == cnpj_petrobras) &
    (bpp["CD_CONTA"] == "2.01")
]

# Conferir se encontramos os registros corretos
print("Registros de Ativo Circulante:", len(ativo))
print("Registros de Passivo Circulante:", len(passivo))

if len(ativo) != 1 or len(passivo) != 1:
    raise ValueError(
        "Esperávamos um registro para cada conta. "
        "Verifique os resultados."
    )

ativo_circulante = ativo["VALOR_REAIS"].iloc[0]
passivo_circulante = passivo["VALOR_REAIS"].iloc[0]

if passivo_circulante <= 0:
    raise ValueError("Passivo circulante inválido.")

# Calcular Liquidez Corrente
liquidez_corrente = (
    ativo_circulante / passivo_circulante
)

print("\nPETROBRAS - LIQUIDEZ CORRENTE 2025")
print("-" * 45)
print(f"Ativo Circulante: R$ {ativo_circulante:,.2f}")
print(f"Passivo Circulante: R$ {passivo_circulante:,.2f}")
print("-" * 45)
print(f"LIQUIDEZ CORRENTE: {liquidez_corrente:.2f}")


Registros de Ativo Circulante: 1
Registros de Passivo Circulante: 1

PETROBRAS - LIQUIDEZ CORRENTE 2025
---------------------------------------------
Ativo Circulante: R$ 140,026,000,000.00
Passivo Circulante: R$ 198,368,000,000.00
---------------------------------------------
LIQUIDEZ CORRENTE: 0.71


In [13]:

# Filtrar o passivo consolidado da Petrobras
balanco_petrobras = bpp[
    bpp["CNPJ_CIA"] == cnpj_petrobras
].copy()

# Localizar contas de Patrimônio Líquido
patrimonio = balanco_petrobras[
    balanco_petrobras["CD_CONTA"].str.match(
        r"^2\.03(\.|$)",
        na=False
    )
]

# Exibir contas principais e respectivos valores
display(
    patrimonio[
        [
            "DT_FIM_EXERC",
            "CD_CONTA",
            "DS_CONTA",
            "VALOR_REAIS"
        ]
    ].head(25)
)


,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VALOR_REAIS
58963,2025-12-31,2.03,Patrimônio Líquido Consolidado,4.175870e+11
58965,2025-12-31,2.03.01,Capital Social Realizado,2.054320e+11
58967,2025-12-31,2.03.02,Reservas de Capital,3.106000e+09
58969,2025-12-31,2.03.02.01,Ágio na Emissão de Ações,0.000000e+00
58971,2025-12-31,2.03.02.02,Reserva Especial de Ágio na Incorporação,0.000000e+00
58973,2025-12-31,2.03.02.03,Alienação de Bônus de Subscrição,0.000000e+00
58975,2025-12-31,2.03.02.04,Opções Outorgadas,0.000000e+00
58977,2025-12-31,2.03.02.05,Ações em Tesouraria,0.000000e+00
58979,2025-12-31,2.03.02.06,Adiantamento para Futuro Aumento de Capital,0.000000e+00
58981,2025-12-31,2.03.03,Reservas de Reavaliação,0.000000e+00


In [14]:

import pandas as pd
import zipfile
from io import BytesIO

# Ler o balanço consolidado, sem excluir
# os dados comparativos do exercício anterior
with zipfile.ZipFile(BytesIO(resposta.content)) as zip_cvm:
    with zip_cvm.open(
        "dfp_cia_aberta_BPP_con_2025.csv"
    ) as arquivo:
        bpp_completo = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            dtype={
                "CNPJ_CIA": str,
                "CD_CONTA": str
            },
            low_memory=False
        )

# Filtrar a Petrobras
bpp_petrobras = bpp_completo[
    bpp_completo["CNPJ_CIA"] == cnpj_petrobras
].copy()

# Selecionar a versão mais recente
ultima_versao = bpp_petrobras.groupby(
    ["CNPJ_CIA", "DT_REFER"]
)["VERSAO"].transform("max")

bpp_petrobras = bpp_petrobras[
    bpp_petrobras["VERSAO"] == ultima_versao
].copy()

# Filtrar patrimônio líquido e não controladores
contas_pl = bpp_petrobras[
    bpp_petrobras["CD_CONTA"].isin([
        "2.03",
        "2.03.09"
    ])
].copy()

# Converter os valores para reais
fatores = {
    "MIL": 1000,
    "UNIDADE": 1,
    "REAL": 1
}

escala = (
    contas_pl["ESCALA_MOEDA"]
    .str.strip()
    .str.upper()
)

contas_pl["VALOR_REAIS"] = (
    pd.to_numeric(contas_pl["VL_CONTA"], errors="raise")
    * escala.map(fatores)
)

display(
    contas_pl[
        [
            "DT_REFER",
            "DT_FIM_EXERC",
            "ORDEM_EXERC",
            "CD_CONTA",
            "DS_CONTA",
            "VALOR_REAIS"
        ]
    ]
)


,DT_REFER,DT_FIM_EXERC,ORDEM_EXERC,CD_CONTA,DS_CONTA,VALOR_REAIS
58962,2025-12-31,2024-12-31,PENÚLTIMO,2.03,Patrimônio Líquido Consolidado,3.675140e+11
58963,2025-12-31,2025-12-31,ÚLTIMO,2.03,Patrimônio Líquido Consolidado,4.175870e+11
59010,2025-12-31,2024-12-31,PENÚLTIMO,2.03.09,Participação dos Acionistas Não Controladores,1.508000e+09
59011,2025-12-31,2025-12-31,ÚLTIMO,2.03.09,Participação dos Acionistas Não Controladores,1.801000e+09


In [15]:

# Organizar patrimônio por data e conta
tabela_pl = contas_pl.pivot(
    index="DT_FIM_EXERC",
    columns="CD_CONTA",
    values="VALOR_REAIS"
)

# Calcular patrimônio dos controladores
tabela_pl["PL_CONTROLADORES"] = (
    tabela_pl["2.03"] - tabela_pl["2.03.09"]
)

pl_2024 = tabela_pl.loc[
    "2024-12-31", "PL_CONTROLADORES"
]

pl_2025 = tabela_pl.loc[
    "2025-12-31", "PL_CONTROLADORES"
]

# Patrimônio líquido médio
pl_medio = (pl_2024 + pl_2025) / 2

# Lucro atribuído aos controladores
conta_lucro_controladores = petrobras[
    petrobras["CD_CONTA"] == "3.11.01"
]

if len(conta_lucro_controladores) != 1:
    raise ValueError("Conta 3.11.01 não encontrada unicamente")

lucro_controladores = (
    conta_lucro_controladores["VALOR_REAIS"].iloc[0]
)

# Calcular ROE
roe = (lucro_controladores / pl_medio) * 100

print("PETROBRAS - ROE 2025")
print("-" * 45)
print(f"PL controladores 2024: R$ {pl_2024:,.2f}")
print(f"PL controladores 2025: R$ {pl_2025:,.2f}")
print(f"PL médio: R$ {pl_medio:,.2f}")
print(f"Lucro controladores: R$ {lucro_controladores:,.2f}")
print("-" * 45)
print(f"ROE: {roe:.2f}%")


PETROBRAS - ROE 2025
---------------------------------------------
PL controladores 2024: R$ 366,006,000,000.00
PL controladores 2025: R$ 415,786,000,000.00
PL médio: R$ 390,896,000,000.00
Lucro controladores: R$ 110,129,000,000.00
---------------------------------------------
ROE: 28.17%


In [16]:

import pandas as pd
import zipfile
from io import BytesIO

with zipfile.ZipFile(BytesIO(resposta.content)) as zip_cvm:
    arquivos_dfc = [
        nome for nome in zip_cvm.namelist()
        if "DFC_MI_con_2025.csv" in nome
    ]

    if not arquivos_dfc:
        raise ValueError("DFC indireta consolidada não encontrada.")

    with zip_cvm.open(arquivos_dfc[0]) as arquivo:
        dfc = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            dtype={
                "CNPJ_CIA": str,
                "CD_CONTA": str
            },
            low_memory=False
        )

dfc = dfc[
    (dfc["CNPJ_CIA"] == cnpj_petrobras) &
    (dfc["ORDEM_EXERC"] == "ÚLTIMO")
].copy()

# Selecionar a versão mais recente
dfc = dfc[
    dfc["VERSAO"] == dfc["VERSAO"].max()
].copy()

# Normalizar valores
fatores = {
    "MIL": 1000,
    "UNIDADE": 1,
    "REAL": 1
}

escala = dfc["ESCALA_MOEDA"].str.strip().str.upper()

if not escala.isin(fatores).all():
    raise ValueError("Escala monetária desconhecida")

dfc["VALOR_REAIS"] = (
    pd.to_numeric(dfc["VL_CONTA"], errors="raise")
    * escala.map(fatores)
)

# Localizar as contas referentes à depreciação
depreciacao = dfc[
    dfc["DS_CONTA"].str.contains(
        "deprecia|amortiza|exaust",
        case=False,
        na=False
    )
]

display(
    depreciacao[
        ["CD_CONTA", "DS_CONTA", "VALOR_REAIS"]
    ]
)


,CD_CONTA,DS_CONTA,VALOR_REAIS
28875,6.01.01.04,"Depreciação, depleção e amortização",8.438800e+10
28971,6.03.03,Amortizações de principal - financiamentos,-1.862900e+10
28973,6.03.04,Amortizações de juros - financiamentos,-1.031100e+10
28981,6.03.08,Amortizações de arrendamentos,-5.243700e+10


In [17]:

# Resultado antes do financeiro e dos tributos
ebit_registros = petrobras[
    petrobras["CD_CONTA"] == "3.05"
]

# Depreciação, depleção e amortização
dda_registros = dfc[
    dfc["CD_CONTA"] == "6.01.01.04"
]

if len(ebit_registros) != 1 or len(dda_registros) != 1:
    raise ValueError(
        "Não foi encontrada uma conta única "
        "para EBIT ou depreciação."
    )

ebit = ebit_registros["VALOR_REAIS"].iloc[0]
dda = dda_registros["VALOR_REAIS"].iloc[0]

ebitda = ebit + dda
margem_ebitda = ebitda / receita * 100

print("PETROBRAS - EBITDA 2025")
print("-" * 45)
print(f"EBIT: R$ {ebit:,.2f}")
print(f"Depreciação e amortização: R$ {dda:,.2f}")
print(f"EBITDA: R$ {ebitda:,.2f}")
print(f"Margem EBITDA: {margem_ebitda:.2f}%")

# Conferência com o EBITDA divulgado
ebitda_divulgado = 230_016_000_000

assert abs(ebitda - ebitda_divulgado) < 1, (
    "EBITDA diferente do valor divulgado."
)

print("Conciliação do EBITDA: OK")


PETROBRAS - EBITDA 2025
---------------------------------------------
EBIT: R$ 145,628,000,000.00
Depreciação e amortização: R$ 84,388,000,000.00
EBITDA: R$ 230,016,000,000.00
Margem EBITDA: 46.23%
Conciliação do EBITDA: OK
